# 02zz — Master Cleanup (limpieza experimental)

Genera **3 versiones de master** con distinta agresividad de limpieza,
todas como superset de eliminaciones (V3 ⊆ V2 ⊆ V1). La master original
permanece intacta como "control" para experimentación en Fase 4.

## Estrategia
- **V1 Conservadora**: solo elimina features muertas (>95% missing, std=0/cuasi-constantes).
- **V2 Intermedia**: V1 + pares con correlación ≥ 0.99 (redundancia evidente).
- **V3 Agresiva**: V2 + pares con correlación 0.95–0.99 + redundancias moderadas.

## Limpieza base aplicada a las 3 versiones
- Targets `churn_14d` / `churn_30d` → bool.
- Columnas `*_days_ago` / `*_days_since_*` con min == -1: clipeadas a [0, ∞)
  (artefacto de redondeo documentado en 02z).

## Re-evaluación runtime
Las listas son declarativas. Si una feature listada para eliminar en CORR_95
tenía como par una feature ya eliminada en CORR_99, la feature de CORR_95
**se retira de la lista** (su par ya no existe → ya no hay redundancia).

Caso aplicado: `arena_fights_total` (par con `fights_pvp_total`, ya en CORR_99).

## Outputs
- `data/data_qc/master_table_v1_conservative.parquet`
- `data/data_qc/master_table_v2_intermediate.parquet`
- `data/data_qc/master_table_v3_aggressive.parquet`
- `informes/fase1_cleaning/master_cleanup/` (report, summary, dropped_columns_v[1|2|3].csv, version_comparison.csv, HTML)

## La master original NO se toca.


In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
import json
from pathlib import Path
from datetime import datetime

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 240)


In [2]:
# [SETUP] Paths y constantes
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'master_cleanup'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)

MASTER_PATH = DATA_QC / 'master_table_cutoff.parquet'
V1_PATH = DATA_QC / 'master_table_cutoff_v1_conservative.parquet'
V2_PATH = DATA_QC / 'master_table_cutoff_v2_intermediate.parquet'
V3_PATH = DATA_QC / 'master_table_cutoff_v3_aggressive.parquet'

# N_SAMPLE se lee dinámicamente del master tras cargarlo (cell 4)

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MASTER_PATH existe: {MASTER_PATH.exists()}")


PROJECT_ROOT: /Users/jezquerro/Documents/tfg
MASTER_PATH existe: True


## BLOQUE 2 — Carga y limpieza base

In [3]:
# [EXEC] 2.1 Cargar master original (read-only desde aquí)
master = pd.read_parquet(MASTER_PATH)
N_SAMPLE = master.shape[0]  # leído dinámicamente
assert N_SAMPLE > 20_000, f'Sample sospechosamente pequeño: {N_SAMPLE}'
assert master.shape[1] >= 100, f'Master con muy pocas columnas: {master.shape}'
print(f'Master cargada: shape={master.shape}, N_SAMPLE={N_SAMPLE:,}')
assert master.columns[0] == 'user_id'
assert master.columns[-2] == 'churn_14d'
assert master.columns[-1] == 'churn_30d'

print(f'Master original: {master.shape}')
log_step('EXEC', f'2.1 master cargada: {master.shape}')

# 2.2 Limpieza base sobre copia
master_base = master.copy()

# A) Targets a bool
master_base['churn_14d'] = master_base['churn_14d'].astype(bool)
master_base['churn_30d'] = master_base['churn_30d'].astype(bool)

# B) Clipear columnas *_days_ago / *_days_since_* con min == -1
days_cols = [c for c in master_base.columns
             if (c.endswith('_days_ago') or '_days_since_' in c)
             and pd.api.types.is_numeric_dtype(master_base[c])]
clipped_cols = []
for col in days_cols:
    s = master_base[col]
    s_no_sentinel = s[s != 9999]
    if len(s_no_sentinel) == 0:
        continue
    if s_no_sentinel.min() == -1:
        master_base[col] = master_base[col].clip(lower=0)
        clipped_cols.append(col)

print(f'\nColumnas clipeadas a [0, ∞): {len(clipped_cols)}')
for c in clipped_cols:
    print(f'  {c}')
log_step('EXEC', f'2.2.A targets convertidos a bool')
log_step('EXEC', f'2.2.B {len(clipped_cols)} columnas días clipeadas a [0, ∞)')

# C) Validar que la base sigue íntegra
assert master_base.shape == (N_SAMPLE, master.shape[1]), f'shape mismatch: {master_base.shape} vs ({N_SAMPLE}, {master.shape[1]})'
assert master_base['churn_14d'].dtype == bool
assert master_base['churn_30d'].dtype == bool
assert master_base['user_id'].is_unique
print(f'\nmaster_base validada: {master_base.shape}, targets bool, user_id único')


Master cargada: shape=(25200, 115), N_SAMPLE=25,200
Master original: (25200, 115)
[EXEC] 13:29:34 — 2.1 master cargada: (25200, 115)

Columnas clipeadas a [0, ∞): 0
[EXEC] 13:29:34 — 2.2.A targets convertidos a bool
[EXEC] 13:29:34 — 2.2.B 0 columnas días clipeadas a [0, ∞)

master_base validada: (25200, 115), targets bool, user_id único


## BLOQUE 3 — Listas declarativas de eliminación

In [4]:
# [EXEC] 3.1-3.4 Listas de eliminación (declarativas)

# 3.1 Lista A — High missing (>95%)
HIGH_MISSING = [
    'iap_first_subscription_days_ago',
    'iap_first_consumable_days_ago',
]

# 3.2 Lista B — Cuasi-constantes
QUASI_CONSTANT = [
    'tutorial_done',
    'user_has_corrupted_dates',
    'user_template_item_stats_augment_update_done',
    'char_is_customized',
]

# 3.2.1 Lista B' — Target leakage (drop SIEMPRE en v1/v2/v3)
# Estos campos derivan directamente de last_login_dt, que es el origen de los
# targets churn_14d/churn_30d. Tenerlos como features es leakage tautológico
# (AUC≈1.0). Se descartan en todas las versiones, no solo en v2/v3.
TARGET_LEAKAGE = [
    # Derivados directos del target (last_login_dt)
    'user_last_login_date',          # unix timestamp = ground truth del target
    'user_last_login_days_ago',      # (CUTOFF - last_login).days
    'user_days_since_last_login',    # (CUTOFF - last_login).clip(0)
    # Snapshot post-cutoff (estado del usuario en REFERENCE_DATE, no en CUTOFF)
    # — todos se actualizan en login y por tanto reflejan actividad post-corte
    'user_game_version',             # se actualiza en cada login → valor post-cutoff
    'user_gold',                     # balance de oro al último login (post-cutoff)
    'user_gems',                     # balance de gemas al último login
    'user_dark_steel',               # balance de dark steel al último login
    'user_runes',                    # balance de runas al último login
    'user_current_character',        # personaje activo al último login
    'user_current_session',          # ID sesión actual al último login
    'user_num_logins',               # contador lifetime, incluye logins post-cutoff
    'user_last_completed_tutorial_block',  # progreso al último login
    'user_updated_at',               # timestamp de última actualización (post-cutoff)
]

# 3.3 Lista C — Correlación >= 0.99 (drop el menos canónico de cada par)
CORR_99_DROP = [
    'fights_pvp_total',
    'arena_gold_earned',
    'dark_steel_last_balance',
    'iap_consumables_days_since_last',
    'feedback_days_since_last',
    'char_talent_total_max',
    'tx_count_last_30d',
    'concept_2_count',
    'iap_days_since_subscription_end',
    'char_first_created_days_ago',
]

# 3.4 Lista D — Correlación 0.95–0.99
# Nota: arena_fights_total se RETIRA de la lista porque su par fights_pvp_total
# ya cae en CORR_99_DROP, por tanto ya no es redundante.
CORR_95_DROP = [
    'arena_fights_as_attacker',
    'arena_win_rate_as_attacker',
    'device_platform_count',
    'char_attack_defense_sum_max',
    'tx_first_days_ago',
    'is_google_play',
    'user_player_lifespan_days',
    'char_talent_spent_max',
    'iap_gems_packs_count',
    'char_defense_max',
    'arena_wins_total',
    'device_first_seen_days_ago',
    # 'arena_fights_total',  # RETIRADA: su par fights_pvp_total cae en CORR_99
    'gems_tx_count',
    'tx_last_days_ago',
    'iap_has_monthly',
    'items_first_item_days_ago',
    'concept_54_count',
    'arena_losses_total',
    'gems_total_spent',
    'items_unique_definitions',
]

# 3.5 Validación de existencia (warning si falta alguna)
def validate_list(name, lst):
    missing = [c for c in lst if c not in master_base.columns]
    if missing:
        log_step('WARNING', f'{name}: cols no encontradas (se ignoran): {missing}')
        lst[:] = [c for c in lst if c in master_base.columns]
    log_step('EXEC', f'{name}: {len(lst)} cols a eliminar')
    return lst

HIGH_MISSING = validate_list('HIGH_MISSING', HIGH_MISSING)
QUASI_CONSTANT = validate_list('QUASI_CONSTANT', QUASI_CONSTANT)
TARGET_LEAKAGE = validate_list('TARGET_LEAKAGE', TARGET_LEAKAGE)
CORR_99_DROP = validate_list('CORR_99_DROP', CORR_99_DROP)
CORR_95_DROP = validate_list('CORR_95_DROP', CORR_95_DROP)

print(f'\n=== Tamaños finales (tras validación) ===')
print(f'  HIGH_MISSING:    {len(HIGH_MISSING)}')
print(f'  QUASI_CONSTANT:  {len(QUASI_CONSTANT)}')
print(f'  TARGET_LEAKAGE:  {len(TARGET_LEAKAGE)} (drop forzado en TODAS las versiones)')
print(f'  CORR_99_DROP:    {len(CORR_99_DROP)}')
print(f'  CORR_95_DROP:    {len(CORR_95_DROP)} (arena_fights_total retirada manualmente)')


[EXEC] 13:29:34 — HIGH_MISSING: 2 cols a eliminar
[EXEC] 13:29:34 — QUASI_CONSTANT: 4 cols a eliminar
[EXEC] 13:29:34 — TARGET_LEAKAGE: 13 cols a eliminar
[WARNING] 13:29:34 — CORR_99_DROP: cols no encontradas (se ignoran): ['fights_pvp_total', 'arena_gold_earned', 'dark_steel_last_balance', 'tx_count_last_30d', 'concept_2_count']
[EXEC] 13:29:34 — CORR_99_DROP: 5 cols a eliminar
[WARNING] 13:29:34 — CORR_95_DROP: cols no encontradas (se ignoran): ['arena_fights_as_attacker', 'arena_win_rate_as_attacker', 'tx_first_days_ago', 'arena_wins_total', 'gems_tx_count', 'tx_last_days_ago', 'concept_54_count', 'arena_losses_total', 'gems_total_spent']
[EXEC] 13:29:34 — CORR_95_DROP: 11 cols a eliminar

=== Tamaños finales (tras validación) ===
  HIGH_MISSING:    2
  QUASI_CONSTANT:  4
  TARGET_LEAKAGE:  13 (drop forzado en TODAS las versiones)
  CORR_99_DROP:    5
  CORR_95_DROP:    11 (arena_fights_total retirada manualmente)


## BLOQUE 4 — Generar las 3 versiones

In [5]:
# [EXEC] 4.1 Versión 1 — Conservadora
to_drop_v1 = HIGH_MISSING + QUASI_CONSTANT + TARGET_LEAKAGE
master_v1 = master_base.drop(columns=to_drop_v1)

assert master_v1.shape[0] == N_SAMPLE
assert master_v1['user_id'].is_unique
assert master_v1.columns[-1] == 'churn_30d'
assert master_v1.columns[-2] == 'churn_14d'
assert len(master_v1.columns) == master.shape[1] - len(to_drop_v1)

print(f'V1 Conservadora: {master_v1.shape} (drop {len(to_drop_v1)})')
log_step('EXEC', f'V1 Conservadora: {master_v1.shape}')


V1 Conservadora: (25200, 96) (drop 19)
[EXEC] 13:29:34 — V1 Conservadora: (25200, 96)


In [6]:
# [EXEC] 4.2 Versión 2 — Intermedia
to_drop_v2 = to_drop_v1 + CORR_99_DROP
master_v2 = master_base.drop(columns=to_drop_v2)

assert master_v2.shape[0] == N_SAMPLE
assert master_v2['user_id'].is_unique
assert master_v2.columns[-1] == 'churn_30d'
assert master_v2.columns[-2] == 'churn_14d'
assert len(master_v2.columns) == master.shape[1] - len(to_drop_v2)

print(f'V2 Intermedia: {master_v2.shape} (drop {len(to_drop_v2)})')
log_step('EXEC', f'V2 Intermedia: {master_v2.shape}')


V2 Intermedia: (25200, 91) (drop 24)
[EXEC] 13:29:34 — V2 Intermedia: (25200, 91)


In [7]:
# [EXEC] 4.3 Versión 3 — Agresiva
to_drop_v3 = to_drop_v2 + CORR_95_DROP
master_v3 = master_base.drop(columns=to_drop_v3)

assert master_v3.shape[0] == N_SAMPLE
assert master_v3['user_id'].is_unique
assert master_v3.columns[-1] == 'churn_30d'
assert master_v3.columns[-2] == 'churn_14d'
assert len(master_v3.columns) == master.shape[1] - len(to_drop_v3)

print(f'V3 Agresiva: {master_v3.shape} (drop {len(to_drop_v3)})')
log_step('EXEC', f'V3 Agresiva: {master_v3.shape}')


V3 Agresiva: (25200, 80) (drop 35)
[EXEC] 13:29:34 — V3 Agresiva: (25200, 80)


In [8]:
# [EXEC] 4.4 Validar superset (V3 ⊆ V2 ⊆ V1)
cols_v1 = set(master_v1.columns)
cols_v2 = set(master_v2.columns)
cols_v3 = set(master_v3.columns)

assert cols_v3.issubset(cols_v2), f'V3 no es subset de V2: extra={cols_v3-cols_v2}'
assert cols_v2.issubset(cols_v1), f'V2 no es subset de V1: extra={cols_v2-cols_v1}'

print(f'Superset OK: V3 ⊆ V2 ⊆ V1')
print(f'  Cardinalidades: V1={len(cols_v1)}, V2={len(cols_v2)}, V3={len(cols_v3)}')

# Tasas de churn (deben ser idénticas — no se eliminan filas)
churn_14d_orig = master['churn_14d'].mean() * 100
churn_30d_orig = master['churn_30d'].mean() * 100
churn_14d_v1 = master_v1['churn_14d'].mean() * 100
churn_30d_v1 = master_v1['churn_30d'].mean() * 100
churn_14d_v2 = master_v2['churn_14d'].mean() * 100
churn_30d_v2 = master_v2['churn_30d'].mean() * 100
churn_14d_v3 = master_v3['churn_14d'].mean() * 100
churn_30d_v3 = master_v3['churn_30d'].mean() * 100

assert abs(churn_14d_v1 - churn_14d_orig) < 1e-6
assert abs(churn_30d_v1 - churn_30d_orig) < 1e-6
assert abs(churn_14d_v3 - churn_14d_orig) < 1e-6
print(f'\nTasas de churn preservadas en las 3 versiones')
print(f'  Churn 14d: {churn_14d_orig:.4f}%')
print(f'  Churn 30d: {churn_30d_orig:.4f}%')

log_step('EXEC', '4.4 superset y tasas validados')


Superset OK: V3 ⊆ V2 ⊆ V1
  Cardinalidades: V1=96, V2=91, V3=80

Tasas de churn preservadas en las 3 versiones
  Churn 14d: 33.9563%
  Churn 30d: 47.4246%
[EXEC] 13:29:34 — 4.4 superset y tasas validados


## BLOQUE 5 — Guardado

In [9]:
# [EXEC] 5 Guardado
t0 = time.time()
master_v1.to_parquet(V1_PATH, engine='pyarrow', compression='snappy', index=False)
master_v2.to_parquet(V2_PATH, engine='pyarrow', compression='snappy', index=False)
master_v3.to_parquet(V3_PATH, engine='pyarrow', compression='snappy', index=False)
print(f'Guardado en {time.time()-t0:.2f}s')

# Verificar tras guardar
sizes = {}
for name, path in [('v1', V1_PATH), ('v2', V2_PATH), ('v3', V3_PATH)]:
    df = pd.read_parquet(path)
    size_mb = path.stat().st_size / 1e6
    sizes[name] = size_mb
    log_step('EXEC', f'{name}: shape={df.shape}, size={size_mb:.2f} MB')

print(f'\nTamaños:')
print(f'  Original: {MASTER_PATH.stat().st_size/1e6:.2f} MB')
for name, sz in sizes.items():
    print(f'  {name:>8}: {sz:.2f} MB')


Guardado en 0.13s


[EXEC] 13:29:35 — v1: shape=(25200, 96), size=2.71 MB


[EXEC] 13:29:35 — v2: shape=(25200, 91), size=2.66 MB


[EXEC] 13:29:35 — v3: shape=(25200, 80), size=2.30 MB

Tamaños:
  Original: 4.08 MB
        v1: 2.71 MB
        v2: 2.66 MB
        v3: 2.30 MB


## BLOQUE 6 — Documentación

In [10]:
# [REPORT] 6.1 dropped_columns_v[1|2|3].csv

# Justificación columna a columna
JUSTIFICATIONS = {
    # HIGH_MISSING
    'iap_first_subscription_days_ago': ('>95% missing (99.14%)', 'HIGH_MISSING'),
    'iap_first_consumable_days_ago': ('>95% missing (99.05%)', 'HIGH_MISSING'),
    # QUASI_CONSTANT
    'tutorial_done': ('Constante (std=0)', 'QUASI_CONSTANT'),
    'user_has_corrupted_dates': ('Constante (std=0)', 'QUASI_CONSTANT'),
    'user_template_item_stats_augment_update_done': ('Constante (std=0)', 'QUASI_CONSTANT'),
    'char_is_customized': ('Cuasi-constante (~99.998% True)', 'QUASI_CONSTANT'),
    # CORR_99_DROP
    'fights_pvp_total': ('Corr 1.000 con arena_fights_as_attacker (mantener arena_fights_as_attacker)', 'CORR_99_DROP'),
    'user_last_login_date': ('Corr -0.9999 con user_days_since_last_login (mantener days_since)', 'CORR_99_DROP'),
    'arena_gold_earned': ('Corr 0.9998 con arena_dark_steel_earned (mantener dark_steel)', 'CORR_99_DROP'),
    'dark_steel_last_balance': ('Corr 0.9997 con user_dark_steel (mantener user_dark_steel)', 'CORR_99_DROP'),
    'iap_consumables_days_since_last': ('Corr -0.9997 con iap_has_consumables (mantener bool)', 'CORR_99_DROP'),
    'feedback_days_since_last': ('Corr -0.9995 con feedback_has_any (mantener bool)', 'CORR_99_DROP'),
    'char_talent_total_max': ('Corr 0.9995 con char_level_max (mantener level)', 'CORR_99_DROP'),
    'tx_count_last_30d': ('Corr 0.9994 con tx_count_total (mantener total)', 'CORR_99_DROP'),
    'concept_2_count': ('Corr 0.9994 con arena_wins_total (mantener wins)', 'CORR_99_DROP'),
    'iap_days_since_subscription_end': ('Corr -0.9993 con iap_has_subscription_ever (mantener bool)', 'CORR_99_DROP'),
    'char_first_created_days_ago': ('Corr 0.9993 con user_created_at_days_ago (mantener canónica user_)', 'CORR_99_DROP'),
    # CORR_95_DROP
    'arena_fights_as_attacker': ('Corr 0.998 con arena_wins_total', 'CORR_95_DROP'),
    'arena_win_rate_as_attacker': ('Corr 0.994 con arena_win_rate (mantener general)', 'CORR_95_DROP'),
    'device_platform_count': ('Corr -0.996 con device_days_since_last_active (mantener days_since)', 'CORR_95_DROP'),
    'char_attack_defense_sum_max': ('Suma redundante con char_attack_max y char_defense_max', 'CORR_95_DROP'),
    'tx_first_days_ago': ('Corr 0.994 con fights_first_days_ago (mantener fights)', 'CORR_95_DROP'),
    'is_google_play': ('Corr -0.986 con device_has_ios (mantener device_has_ios, más explícito)', 'CORR_95_DROP'),
    'user_player_lifespan_days': ('Corr 0.985 con user_created_at_days_ago (mantener created_at)', 'CORR_95_DROP'),
    'char_talent_spent_max': ('Corr 0.984 con char_level_max', 'CORR_95_DROP'),
    'iap_gems_packs_count': ('Corr 0.983 con iap_consumables_count', 'CORR_95_DROP'),
    'char_defense_max': ('Corr 0.980 con char_attack_max (mantener attack; defensa preservada en items_attack_defense_ratio)', 'CORR_95_DROP'),
    'arena_wins_total': ('Corr 0.979 con arena_fights_total (si juega gana, redundante)', 'CORR_95_DROP'),
    'device_first_seen_days_ago': ('Corr 0.978 con user_created_at_days_ago', 'CORR_95_DROP'),
    'gems_tx_count': ('Corr 0.978 con other_concepts_count', 'CORR_95_DROP'),
    'tx_last_days_ago': ('Corr 0.975 con user_days_since_last_login', 'CORR_95_DROP'),
    'iap_has_monthly': ('Corr 0.971 con iap_has_subscription_ever', 'CORR_95_DROP'),
    'items_first_item_days_ago': ('Corr 0.969 con char_first_created_days_ago / user_created_at', 'CORR_95_DROP'),
    'concept_54_count': ('Corr 0.966 con tx_days_active', 'CORR_95_DROP'),
    'arena_losses_total': ('Corr 0.966 con arena_fights_as_defender', 'CORR_95_DROP'),
    'gems_total_spent': ('Corr 0.954 con gems_total_earned (mantener earned como proxy de actividad)', 'CORR_95_DROP'),
    'items_unique_definitions': ('Corr 0.952 con items_total_instances', 'CORR_95_DROP'),
}


def make_dropped_csv(cols_dropped, path):
    rows = []
    for c in cols_dropped:
        reason, source = JUSTIFICATIONS.get(c, ('(sin justificación documentada)', '?'))
        rows.append({'column': c, 'reason': reason, 'source_list': source})
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df


df_v1 = make_dropped_csv(to_drop_v1, REPORT_DIR / 'dropped_columns_v1.csv')
df_v2 = make_dropped_csv(to_drop_v2, REPORT_DIR / 'dropped_columns_v2.csv')
df_v3 = make_dropped_csv(to_drop_v3, REPORT_DIR / 'dropped_columns_v3.csv')

print(f'dropped_columns_v1.csv: {len(df_v1)} cols')
print(f'dropped_columns_v2.csv: {len(df_v2)} cols')
print(f'dropped_columns_v3.csv: {len(df_v3)} cols')
log_step('REPORT', f'6.1 dropped_columns_v[1|2|3].csv generados')


dropped_columns_v1.csv: 19 cols
dropped_columns_v2.csv: 24 cols
dropped_columns_v3.csv: 35 cols
[REPORT] 13:29:35 — 6.1 dropped_columns_v[1|2|3].csv generados


In [11]:
# [REPORT] 6.2 version_comparison.csv
size_orig = MASTER_PATH.stat().st_size / 1e6

comparison = pd.DataFrame([
    {
        'version': 'original',
        'n_cols': master.shape[1],
        'n_dropped': 0,
        'size_mb': round(size_orig, 2),
        'churn_14d_rate': round(churn_14d_orig / 100, 6),
        'churn_30d_rate': round(churn_30d_orig / 100, 6),
    },
    {
        'version': 'v1_conservative',
        'n_cols': master_v1.shape[1],
        'n_dropped': len(to_drop_v1),
        'size_mb': round(sizes['v1'], 2),
        'churn_14d_rate': round(churn_14d_v1 / 100, 6),
        'churn_30d_rate': round(churn_30d_v1 / 100, 6),
    },
    {
        'version': 'v2_intermediate',
        'n_cols': master_v2.shape[1],
        'n_dropped': len(to_drop_v2),
        'size_mb': round(sizes['v2'], 2),
        'churn_14d_rate': round(churn_14d_v2 / 100, 6),
        'churn_30d_rate': round(churn_30d_v2 / 100, 6),
    },
    {
        'version': 'v3_aggressive',
        'n_cols': master_v3.shape[1],
        'n_dropped': len(to_drop_v3),
        'size_mb': round(sizes['v3'], 2),
        'churn_14d_rate': round(churn_14d_v3 / 100, 6),
        'churn_30d_rate': round(churn_30d_v3 / 100, 6),
    },
])
comparison.to_csv(REPORT_DIR / 'version_comparison.csv', index=False)
print(comparison.to_string(index=False))
log_step('REPORT', f'6.2 version_comparison.csv generado')


        version  n_cols  n_dropped  size_mb  churn_14d_rate  churn_30d_rate
       original     115          0     4.08        0.339563        0.474246
v1_conservative      96         19     2.71        0.339563        0.474246
v2_intermediate      91         24     2.66        0.339563        0.474246
  v3_aggressive      80         35     2.30        0.339563        0.474246
[REPORT] 13:29:35 — 6.2 version_comparison.csv generado


In [12]:
# [REPORT] 6.3 execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

lines = [
    '# Execution Report — 02zz_master_cleanup',
    '',
    f'**Notebook**: notebooks/fase1_cleaning/02zz_master_cleanup.ipynb',
    f'**Fecha**: {now_str}',
    f'**Tiempo de ejecución**: {t_min} min {t_sec} s',
    f'**Master origen**: master_table_cutoff.parquet ({master.shape[0]:,} × {master.shape[1]})',
    '',
    '## Estrategia',
    '',
    '3 versiones de master generadas con distinta agresividad para experimentación',
    'en Fase 4. Master original intacta como control. Las 3 versiones cumplen',
    'V3 ⊆ V2 ⊆ V1 (cada una superset de eliminaciones de la anterior).',
    '',
    '---',
    '',
    '## Limpieza base (aplicada a las 3 versiones)',
    '',
    f'- Targets convertidos a bool: `churn_14d`, `churn_30d`',
    f'- {len(clipped_cols)} columnas `*_days_ago`/`*_days_since_*` clipeadas a [0, ∞):',
]
for c in clipped_cols:
    lines.append(f'  - `{c}`')
lines += [
    '',
    'Razón del clipeo: artefacto de redondeo documentado en 02z (REFERENCE_DATE',
    'hardcoded a `2026-04-04 00:00 UTC` vs fecha efectiva del dataset',
    '`2026-04-04 18:23:31`). Filas con timestamp del mismo día daban `_days_ago = -1`',
    'que se reinterpretan como 0 en este notebook.',
    '',
    '---',
    '',
    '## Versiones generadas',
    '',
    '| Versión | Cols | Eliminadas | Tamaño | Razón |',
    '|---|---|---|---|---|',
    f'| original | {master.shape[1]} | 0 | {size_orig:.2f} MB | sin limpiar (control) |',
    f'| v1 Conservadora | {master_v1.shape[1]} | {len(to_drop_v1)} | {sizes["v1"]:.2f} MB | High missing + cuasi-constantes |',
    f'| v2 Intermedia | {master_v2.shape[1]} | {len(to_drop_v2)} | {sizes["v2"]:.2f} MB | v1 + correlación >= 0.99 |',
    f'| v3 Agresiva | {master_v3.shape[1]} | {len(to_drop_v3)} | {sizes["v3"]:.2f} MB | v2 + correlación 0.95-0.99 |',
    '',
    '---',
    '',
    '## Validación',
    '',
    f'- [] V3 ⊆ V2 ⊆ V1 (cardinalidades: {len(cols_v1)}, {len(cols_v2)}, {len(cols_v3)})',
    '- [] Asserts de shape pasados en las 3 versiones',
    '- [] user_id único en las 3',
    '- [] Targets en posiciones [-2, -1] en las 3',
    f'- [] Tasas de churn preservadas (14d: {churn_14d_orig:.4f}%, 30d: {churn_30d_orig:.4f}%)',
    '',
    '---',
    '',
    '## Re-evaluación runtime de listas',
    '',
    'Caso aplicado: `arena_fights_total` se RETIRA de CORR_95_DROP porque su par',
    '`fights_pvp_total` ya cae en CORR_99_DROP. Sin esta retirada, ambos miembros',
    'del par redundante hubieran sido eliminados (manteniendo cero, no la',
    'feature canónica).',
    '',
    '---',
    '',
    '## Decisiones documentadas (justificación columna a columna)',
    '',
    '### CORR_99_DROP (correlación >= 0.99)',
    '',
    '| Columna eliminada | Justificación |',
    '|---|---|',
]
for c in CORR_99_DROP:
    reason = JUSTIFICATIONS.get(c, ('?', '?'))[0]
    lines.append(f'| `{c}` | {reason} |')

lines += [
    '',
    '### CORR_95_DROP (correlación 0.95–0.99)',
    '',
    '| Columna eliminada | Justificación |',
    '|---|---|',
]
for c in CORR_95_DROP:
    reason = JUSTIFICATIONS.get(c, ('?', '?'))[0]
    lines.append(f'| `{c}` | {reason} |')

lines += [
    '',
    '---',
    '',
    '## Pasos ejecutados', '',
]
for s in steps_log:
    lines.append(f'- {s}')

lines += [
    '',
    '---',
    '',
    '## TODOs para Fase 4',
    '',
    '- Entrenar baseline (XGBoost/LightGBM o equivalente) con las 4 versiones (original + v1/v2/v3).',
    '- Comparar métricas (AUC, recall@k) y elegir la versión productiva.',
    '- Documentar en memoria del TFG la decisión final con la comparativa.',
    '- Si v3 funciona igual que v1, defendible quedarse con v3 (modelo más ligero).',
    '- Si v1 supera a v3, hay señal en alguna de las features eliminadas — revisar correlación con target específicamente.',
    '',
    '---',
    '',
    '## Archivos generados',
    '',
    '| Archivo | Tipo |',
    '|---|---|',
    '| `data/data_qc/master_table_cutoff_v1_conservative.parquet` | parquet v1 |',
    '| `data/data_qc/master_table_cutoff_v2_intermediate.parquet` | parquet v2 |',
    '| `data/data_qc/master_table_cutoff_v3_aggressive.parquet` | parquet v3 |',
    '| `informes/fase1_cleaning/master_cleanup/execution_report.md` | este informe |',
    '| `informes/fase1_cleaning/master_cleanup/cleanup_summary.md` | resumen ejecutivo |',
    '| `informes/fase1_cleaning/master_cleanup/dropped_columns_v1.csv` | columnas dropeadas v1 |',
    '| `informes/fase1_cleaning/master_cleanup/dropped_columns_v2.csv` | columnas dropeadas v2 |',
    '| `informes/fase1_cleaning/master_cleanup/dropped_columns_v3.csv` | columnas dropeadas v3 |',
    '| `informes/fase1_cleaning/master_cleanup/version_comparison.csv` | tabla comparativa |',
    '| `informes/fase1_cleaning/master_cleanup/02zz_master_cleanup_run.html` | HTML del notebook |',
    '',
]

REPORT_PATH = REPORT_DIR / 'execution_report.md'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'execution_report.md guardado: {REPORT_PATH}')
log_step('REPORT', '6.3 execution_report.md generado')


execution_report.md guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/master_cleanup/execution_report.md
[REPORT] 13:29:35 — 6.3 execution_report.md generado


In [13]:
# [REPORT] 6.4 cleanup_summary.md
lines = [
    '# Resumen ejecutivo — Master Cleanup',
    '',
    '## Cuatro masters disponibles para Fase 4',
    '',
    '| Versión | Cols | Tamaño | Filosofía |',
    '|---|---|---|---|',
    f'| Original (`master_table_cutoff.parquet`) | {master.shape[1]} | {size_orig:.2f} MB | Sin limpiar — control |',
    f'| V1 Conservadora | {master_v1.shape[1]} | {sizes["v1"]:.2f} MB | Solo features muertas (-{len(to_drop_v1)}) |',
    f'| V2 Intermedia | {master_v2.shape[1]} | {sizes["v2"]:.2f} MB | + redundancias evidentes (-{len(to_drop_v2)}) |',
    f'| V3 Agresiva | {master_v3.shape[1]} | {sizes["v3"]:.2f} MB | + redundancias moderadas (-{len(to_drop_v3)}) |',
    '',
    '## Lo que va igual en las 3 versiones',
    '',
    '- Targets `churn_14d` / `churn_30d` convertidos a bool',
    f'- {len(clipped_cols)} columnas días clipeadas a [0, ∞)',
    f'- Sample preservado (126.253 jugadores en las 3)',
    f'- Tasas de churn idénticas (14d: {churn_14d_orig:.2f}%, 30d: {churn_30d_orig:.2f}%)',
    '',
    '## Garantía de superset',
    '',
    f'- `V3 ⊆ V2 ⊆ V1`: cada versión es subset de la anterior (eliminaciones acumulativas)',
    f'- Cardinalidades: V1={len(cols_v1)}, V2={len(cols_v2)}, V3={len(cols_v3)}',
    '',
    '## Re-evaluación runtime',
    '',
    '`arena_fights_total` retirada de la lista CORR_95_DROP porque su par redundante',
    '(`fights_pvp_total`) ya se elimina en CORR_99_DROP. Esto evita que ambos miembros',
    'del par caigan, lo cual hubiera dejado cero features de ese clúster en V3.',
    '',
    '## Próximo paso',
    '',
    'Fase 4: entrenar el mismo modelo (XGBoost/LightGBM baseline) sobre las 4 versiones',
    'y comparar AUC. La que mejor funcione es la productiva. Hipótesis razonables:',
    '',
    '- **v3 ≈ v2 ≈ v1 ≈ original**: redundancias eliminadas no afectan al modelo →',
    '  defendible quedarse con v3 (más ligera, más interpretable).',
    '- **v1 > v3**: alguna feature de las eliminadas en CORR_95 sí aporta señal',
    '  específica → revisar a qué target correlaciona y reincorporar.',
    '- **original > v1**: improbable; las HIGH_MISSING y QUASI_CONSTANT no tienen',
    '  varianza para ayudar al modelo. Si pasa, debugar el pipeline de cleanup.',
    '',
]

SUMMARY_PATH = REPORT_DIR / 'cleanup_summary.md'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'cleanup_summary.md guardado: {SUMMARY_PATH}')
log_step('REPORT', '6.4 cleanup_summary.md generado')


cleanup_summary.md guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/master_cleanup/cleanup_summary.md
[REPORT] 13:29:35 — 6.4 cleanup_summary.md generado


In [14]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START
print('=' * 70)
print(f'RESUMEN FINAL — Notebook 02zz_master_cleanup')
print('=' * 70)
print(f'  Tiempo total                : {int(elapsed//60)}m {int(elapsed%60)}s')
print()
print(f'  Master original             : {master.shape}, {size_orig:.2f} MB')
print(f'  V1 Conservadora             : {master_v1.shape} (drop {len(to_drop_v1)}), {sizes["v1"]:.2f} MB')
print(f'  V2 Intermedia               : {master_v2.shape} (drop {len(to_drop_v2)}), {sizes["v2"]:.2f} MB')
print(f'  V3 Agresiva                 : {master_v3.shape} (drop {len(to_drop_v3)}), {sizes["v3"]:.2f} MB')
print()
print(f'  Superset garantizado        : V3 ⊆ V2 ⊆ V1')
print(f'  Targets bool en las 3       :')
print(f'  Días clipeadas a [0, ∞)     : {len(clipped_cols)}')
print()
print(f'  Tasa churn 14d (todas igual): {churn_14d_orig:.2f}%')
print(f'  Tasa churn 30d (todas igual): {churn_30d_orig:.2f}%')
print()
print(f'  Master original NO modificada (validación final):')
master_check = pd.read_parquet(MASTER_PATH)
print(f'    shape original tras todo = {master_check.shape}')
assert master_check.shape[0] == N_SAMPLE and master_check.shape[1] == master.shape[1], 'Master original alterada — STOP'
print(f'    Master original intacta')
print()
print(f'  Outputs en {REPORT_DIR}:')
print(f'    execution_report.md, cleanup_summary.md')
print(f'    dropped_columns_v[1|2|3].csv, version_comparison.csv')
print('=' * 70)


RESUMEN FINAL — Notebook 02zz_master_cleanup
  Tiempo total                : 0m 0s

  Master original             : (25200, 115), 4.08 MB
  V1 Conservadora             : (25200, 96) (drop 19), 2.71 MB
  V2 Intermedia               : (25200, 91) (drop 24), 2.66 MB
  V3 Agresiva                 : (25200, 80) (drop 35), 2.30 MB

  Superset garantizado        : V3 ⊆ V2 ⊆ V1
  Targets bool en las 3       :
  Días clipeadas a [0, ∞)     : 0

  Tasa churn 14d (todas igual): 33.96%
  Tasa churn 30d (todas igual): 47.42%

  Master original NO modificada (validación final):
    shape original tras todo = (25200, 115)
    Master original intacta

  Outputs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/master_cleanup:
    execution_report.md, cleanup_summary.md
    dropped_columns_v[1|2|3].csv, version_comparison.csv


In [15]:
# [REPORT] Logging dual: HTML
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02zz_master_cleanup.ipynb'
html_path = REPORT_DIR / '02zz_master_cleanup_run.html'
export_notebook_to_html(notebook_path, html_path)

# Sección enriquecida sobre la v3 (la más limpia)
enriched = build_enriched_report_section(
    df_final=master_v3,
    raw_shape=(master.shape[0], master.shape[1]),
    filter_steps=[
        ('Master original', master.shape[1]),
        ('V1 Conservadora', master_v1.shape[1]),
        ('V2 Intermedia', master_v2.shape[1]),
        ('V3 Agresiva', master_v3.shape[1]),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n## Outputs detallados (V3 Agresiva)\n\n' + enriched)
print(f'Enriquecimiento appendeado al execution_report')


HTML generado: 02zz_master_cleanup_run.html (0.4 MB)
Enriquecimiento appendeado al execution_report
